In [ ]:
# ============================================================
# MODEL 2 — IMPORTS
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Project root
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.multiclass_sequence_creation_v2 import (
    SEQUENCE_LENGTH,
    CLASS_MAPPING,
    REVERSE_CLASS_MAPPING,
    create_multiclass_sequences,
    validate_multiclass_sequences,
    calculate_class_weights,
)

In [ ]:
# ============================================================
# LOAD MODEL 2 DATA FOR SEQUENCE CREATION
# ============================================================

import numpy as np
import json
from pathlib import Path


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

scaled_dir = PROJECT_ROOT / "data" / "scaled" / "model2"
model2_dir = PROJECT_ROOT / "data" / "model2"


# ============================================================
# 1. LOAD SCALED MODEL 2 FEATURES
# ============================================================

X_train_scaled = np.load(
    scaled_dir / "X_train_model2_scaled.npy"
)

X_validation_scaled = np.load(
    scaled_dir / "X_validation_model2_scaled.npy"
)

X_test_scaled = np.load(
    scaled_dir / "X_test_model2_scaled.npy"
)


# ============================================================
# 2. LOAD MODEL 2 TARGETS
#    Target = tipo_anomalia
# ============================================================

y_train_model2 = np.load(
    scaled_dir / "y_train_model2.npy",
    allow_pickle=True
)

y_validation_model2 = np.load(
    scaled_dir / "y_validation_model2.npy",
    allow_pickle=True
)

y_test_model2 = np.load(
    scaled_dir / "y_test_model2.npy",
    allow_pickle=True
)


# ============================================================
# 3. LOAD EXACT FEATURE ORDER
# ============================================================

with open(
    scaled_dir / "model2_features.json",
    "r",
    encoding="utf-8"
) as f:
    MODEL2_FINAL_FEATURES = json.load(f)


# ============================================================
# 4. LOAD ORIGINAL MODEL 2 DATAFRAMES
#    Useful for FECHA / traceability / verification
# ============================================================

train_model2 = pd.read_parquet(
    model2_dir / "train_model2.parquet"
)

validation_model2 = pd.read_parquet(
    model2_dir / "validation_model2.parquet"
)

test_model2 = pd.read_parquet(
    model2_dir / "test_model2.parquet"
)


# ============================================================
# 5. BASIC VERIFICATION
# ============================================================

print("=" * 80)
print("MODEL 2 DATA LOADED FOR SEQUENCE CREATION")
print("=" * 80)

print("\nScaled features:")
print(f"X_train_scaled:       {X_train_scaled.shape}")
print(f"X_validation_scaled:  {X_validation_scaled.shape}")
print(f"X_test_scaled:        {X_test_scaled.shape}")

print("\nTargets:")
print(f"y_train_model2:       {y_train_model2.shape}")
print(f"y_validation_model2:  {y_validation_model2.shape}")
print(f"y_test_model2:        {y_test_model2.shape}")

print("\nNumber of Model 2 features:")
print(len(MODEL2_FINAL_FEATURES))

print("\nOriginal Model 2 DataFrames:")
print(f"train_model2:         {train_model2.shape}")
print(f"validation_model2:    {validation_model2.shape}")
print(f"test_model2:          {test_model2.shape}")

print("\nTarget distributions:")

print("\nTRAIN")
print(train_model2["tipo_anomalia"].value_counts(dropna=False))

print("\nVALIDATION")
print(validation_model2["tipo_anomalia"].value_counts(dropna=False))

print("\nTEST")
print(test_model2["tipo_anomalia"].value_counts(dropna=False))


# ============================================================
# 6. ALIGNMENT CHECKS
# ============================================================

assert X_train_scaled.shape[0] == len(y_train_model2), \
    "Train X/y row count mismatch"

assert X_validation_scaled.shape[0] == len(y_validation_model2), \
    "Validation X/y row count mismatch"

assert X_test_scaled.shape[0] == len(y_test_model2), \
    "Test X/y row count mismatch"

assert X_train_scaled.shape[1] == len(MODEL2_FINAL_FEATURES), \
    "Train feature count does not match MODEL2_FINAL_FEATURES"

assert X_validation_scaled.shape[1] == len(MODEL2_FINAL_FEATURES), \
    "Validation feature count does not match MODEL2_FINAL_FEATURES"

assert X_test_scaled.shape[1] == len(MODEL2_FINAL_FEATURES), \
    "Test feature count does not match MODEL2_FINAL_FEATURES"

print("\n" + "=" * 80)
print("ALL MODEL 2 INPUTS ARE ALIGNED")
print("=" * 80)

In [ ]:
# ============================================================
# TRAIN SEQUENCES — MODEL 2
# ============================================================

X_train_m2, y_train_m2 = create_multiclass_sequences(
    X_scaled=X_train_scaled,
    tipo_anomalia=y_train_model2,
    feature_names=MODEL2_FINAL_FEATURES,
    split_name="TRAIN",
    sequence_length=SEQUENCE_LENGTH,
)

print("=" * 80)
print("MODEL 2 — TRAIN SEQUENCES CREATED")
print("=" * 80)

print("X_train_m2:", X_train_m2.shape)
print("y_train_m2:", y_train_m2.shape)
print("Classes:", np.unique(y_train_m2, return_counts=True))

In [ ]:
# ============================================================
# VALIDATION SEQUENCES — MODEL 2
# ============================================================

X_validation_m2, y_validation_m2 = create_multiclass_sequences(
    X_scaled=X_validation_scaled,
    tipo_anomalia=y_validation_model2,
    feature_names=MODEL2_FINAL_FEATURES,
    split_name="VALIDATION",
    sequence_length=SEQUENCE_LENGTH,
)

print("=" * 80)
print("MODEL 2 — VALIDATION SEQUENCES CREATED")
print("=" * 80)

print("X_validation_m2:", X_validation_m2.shape)
print("y_validation_m2:", y_validation_m2.shape)
print("Classes:", np.unique(y_validation_m2, return_counts=True))

In [ ]:
# ============================================================
# TEST SEQUENCES — MODEL 2
# ============================================================

X_test_m2, y_test_m2 = create_multiclass_sequences(
    X_scaled=X_test_scaled,
    tipo_anomalia=y_test_model2,
    feature_names=MODEL2_FINAL_FEATURES,
    split_name="TEST",
    sequence_length=SEQUENCE_LENGTH,
)

print("=" * 80)
print("MODEL 2 — TEST SEQUENCES CREATED")
print("=" * 80)

print("X_test_m2:", X_test_m2.shape)
print("y_test_m2:", y_test_m2.shape)
print("Classes:", np.unique(y_test_m2, return_counts=True))

In [ ]:
# ============================================================
# VALIDATE SEQUENCE ARRAYS
# ============================================================

train_checks = validate_multiclass_sequences(
    X_train_m2,
    y_train_m2,
    MODEL2_FINAL_FEATURES,
)

validation_checks = validate_multiclass_sequences(
    X_validation_m2,
    y_validation_m2,
    MODEL2_FINAL_FEATURES,
)

test_checks = validate_multiclass_sequences(
    X_test_m2,
    y_test_m2,
    MODEL2_FINAL_FEATURES,
)

print("TRAIN:")
print(train_checks)

print("\nVALIDATION:")
print(validation_checks)

print("\nTEST:")
print(test_checks)

In [ ]:
# ============================================================
# CLASS DISTRIBUTION
# ============================================================

def print_class_distribution(y, split_name):
    counts = np.bincount(
        y,
        minlength=len(CLASS_MAPPING),
    )

    print(f"\n{split_name}")
    print("-" * 60)

    for class_id, class_name in REVERSE_CLASS_MAPPING.items():
        print(
            f"{class_id}: "
            f"{class_name:<28} "
            f"{counts[class_id]:>8,}"
        )


print_class_distribution(
    y_train_m2,
    "TRAIN",
)

print_class_distribution(
    y_validation_m2,
    "VALIDATION",
)

print_class_distribution(
    y_test_m2,
    "TEST",
)

In [ ]:
# ============================================================
# SAVE MODEL 2 LSTM SEQUENCES
# ============================================================

from pathlib import Path
import numpy as np

# ------------------------------------------------------------
# Output directory
# ------------------------------------------------------------

sequence_dir = PROJECT_ROOT / "data" / "sequences" / "model2"
sequence_dir.mkdir(parents=True, exist_ok=True)


# ============================================================
# SAVE TRAIN SEQUENCES
# ============================================================

np.save(
    sequence_dir / "train_model2_X.npy",
    X_train_m2
)

np.save(
    sequence_dir / "train_model2_y.npy",
    y_train_m2
)


# ============================================================
# SAVE VALIDATION SEQUENCES
# ============================================================

np.save(
    sequence_dir / "validation_model2_X.npy",
    X_validation_m2
)

np.save(
    sequence_dir / "validation_model2_y.npy",
    y_validation_m2
)


# ============================================================
# SAVE TEST SEQUENCES
# ============================================================

np.save(
    sequence_dir / "test_model2_X.npy",
    X_test_m2
)

np.save(
    sequence_dir / "test_model2_y.npy",
    y_test_m2
)


# ============================================================
# VERIFY SAVED FILES
# ============================================================

print("=" * 80)
print("MODEL 2 — SEQUENCES SAVED")
print("=" * 80)

for path in sorted(sequence_dir.iterdir()):
    print(f"{path.name:<30} {path.stat().st_size / (1024**2):.2f} MB")


# ============================================================
# VERIFY SHAPES
# ============================================================

print("\nShapes:")
print(f"X_train_m2:        {X_train_m2.shape}")
print(f"y_train_m2:        {y_train_m2.shape}")

print(f"X_validation_m2:   {X_validation_m2.shape}")
print(f"y_validation_m2:   {y_validation_m2.shape}")

print(f"X_test_m2:         {X_test_m2.shape}")
print(f"y_test_m2:         {y_test_m2.shape}")

print("\n" + "=" * 80)
print("MODEL 2 SEQUENCES SAVED SUCCESSFULLY")
print("=" * 80)